In [2]:
# ============================================
# 1️⃣ CONECTAR GOOGLE DRIVE
# ============================================

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# ============================================
# 2️⃣ ACCEDER A LA CARPETA "DATOS SECOP ULTIMO AÑO"
# ============================================

import os
import pandas as pd

carpeta = "/content/drive/MyDrive/DATOS SECOP ULTIMO AÑO"
os.chdir(carpeta)

print("📁 Archivos en la carpeta:")
for archivo in os.listdir():
    print(archivo)


📁 Archivos en la carpeta:
D_Categoria.xlsx
D_Modalidad.xlsx
D_Entidad.xlsx
D_Tiempo.xlsx
D_UbiProveedor.xlsx
D_UbiEntidad.xlsx
D_TipoContrato.xlsx
D_Proveedor.xlsx
F_Proceso_parte1.xlsx
F_Proceso_parte2.xlsx


In [ ]:
# ============================================
# 3️⃣ LEER Y NOMBRAR LOS ARCHIVOS (VERSIÓN ROBUSTA)
# ============================================

import os
import pandas as pd
from pathlib import Path

# Lista de archivos esperados (ajusta si los nombres cambian)
expected = {
    "D_Categoria.xlsx": "df_categoria",
    "D_Entidad.xlsx": "df_entidad",
    "D_Modalidad.xlsx": "df_modalidad",
    "D_Proveedor.xlsx": "df_proveedor",
    "D_Tiempo.xlsx": "df_tiempo",
    "D_TipoContrato.xlsx": "df_tipocontrato",
    "D_UbiEntidad.xlsx": "df_ubientidad",
    "D_UbiProveedor.xlsx": "df_ubiproveedor",
    "F_Proceso_parte1.xlsx": "f_proceso_1",
    "F_Proceso_parte2.xlsx": "f_proceso_2"
}

# 1) Mostrar directorio actual y archivos
print("Directorio actual:", os.getcwd())
print("\nArchivos en carpeta:")
for f in sorted(os.listdir()):
    print(" ", f)

# 2) Verificar existencia
missing = [f for f in expected.keys() if not Path(f).exists()]
if missing:
    print("\n❌ Faltan archivos esperados en la carpeta:")
    for f in missing:
        print("  -", f)
    raise FileNotFoundError("Asegúrate de que los archivos estén en la carpeta actual o monta/cambia a la carpeta correcta.")

# 3) Función segura para leer archivos (intenta read_excel, si falla muestra el error y prueba read_csv)
def safe_read(file_path):
    try:
        # intento principal con openpyxl (xlsx)
        df = pd.read_excel(file_path, engine="openpyxl")
        return df, None
    except Exception as e_excel:
        # mostrar error de excel
        print(f"\n⚠️ Error leyendo '{file_path}' con read_excel: {repr(e_excel)}")
        # intentar leer como CSV (fallback)
        try:
            df = pd.read_csv(file_path)
            print(f"ℹ️ Se leyó '{file_path}' como CSV de forma alternativa.")
            return df, None
        except Exception as e_csv:
            print(f"⚠️ También falló leer '{file_path}' como CSV: {repr(e_csv)}")
            return None, (e_excel, e_csv)

# 4) Leer todos los archivos en un diccionario
loaded = {}
errors = {}
for filename, varname in expected.items():
    df, err = safe_read(filename)
    if df is not None:
        loaded[varname] = df
        print(f"\n✅ Cargado: {filename} -> variable: {varname} (shape: {df.shape})")
    else:
        errors[filename] = err
        print(f"\n❌ No se pudo cargar: {filename}")

# 5) Informar si hubo errores
if errors:
    print("\n\n--- RESUMEN: HUBO ERRORES AL LEER ALGUNOS ARCHIVOS ---")
    for fn, err in errors.items():
        print("\nArchivo:", fn)
        print("Errores registrados:", err)
    raise RuntimeError("Corrige los errores de lectura antes de continuar.")

# 6) Asignar variables con nombres originales (solo si se cargaron)
df_categoria    = loaded.get("df_categoria")
df_entidad      = loaded.get("df_entidad")
df_modalidad    = loaded.get("df_modalidad")
df_proveedor    = loaded.get("df_proveedor")
df_tiempo       = loaded.get("df_tiempo")
df_tipocontrato = loaded.get("df_tipocontrato")
df_ubientidad   = loaded.get("df_ubientidad")
df_ubiproveedor = loaded.get("df_ubiproveedor")
f_proceso_1     = loaded.get("f_proceso_1")
f_proceso_2     = loaded.get("f_proceso_2")

# 7) Unir las dos partes de la tabla FACT (si ambas existen)
if f_proceso_1 is None or f_proceso_2 is None:
    raise RuntimeError("Faltan las partes de la tabla F_Proceso. Revisa la carga de F_Proceso_parte1/parte2.")
else:
    df_datos_completos = pd.concat([f_proceso_1, f_proceso_2], ignore_index=True)
    print("\n✅ df_datos_completos creado (concat de las 2 partes). Shape:", df_datos_completos.shape)

# 8) Mostrar un resumen rápido de las tablas cargadas
print("\n\n--- Resumen rápido ---")
for name, var in [
    ("df_categoria", df_categoria),
    ("df_entidad", df_entidad),
    ("df_modalidad", df_modalidad),
    ("df_proveedor", df_proveedor),
    ("df_tiempo", df_tiempo),
    ("df_tipocontrato", df_tipocontrato),
    ("df_ubientidad", df_ubientidad),
    ("df_ubiproveedor", df_ubiproveedor),
    ("df_datos_completos", df_datos_completos)
]:
    print(f"{name}: {'No cargada' if var is None else str(var.shape)}")

# Mostrar primeras filas de la tabla unida
print("\nPrimeras filas de df_datos_completos:")
display(df_datos_completos.head())



Directorio actual: /content/drive/MyDrive/DATOS SECOP ULTIMO AÑO

Archivos en carpeta:
  D_Categoria.xlsx
  D_Entidad.xlsx
  D_Modalidad.xlsx
  D_Proveedor.xlsx
  D_Tiempo.xlsx
  D_TipoContrato.xlsx
  D_UbiEntidad.xlsx
  D_UbiProveedor.xlsx
  F_Proceso_parte1.xlsx
  F_Proceso_parte2.xlsx

✅ Cargado: D_Categoria.xlsx -> variable: df_categoria (shape: (11937, 2))

✅ Cargado: D_Entidad.xlsx -> variable: df_entidad (shape: (18411, 8))

✅ Cargado: D_Modalidad.xlsx -> variable: df_modalidad (shape: (44, 3))


In [ ]:
# ============================================
# 4️⃣ VERIFICAR CONTENIDO DE CADA DATAFRAME
# ============================================

dataframes = {
    "D_Categoria": df_categoria,
    "D_Entidad": df_entidad,
    "D_Modalidad": df_modalidad,
    "D_Proveedor": df_proveedor,
    "D_Tiempo": df_tiempo,
    "D_TipoContrato": df_tipocontrato,
    "D_UbiEntidad": df_ubientidad,
    "D_UbiProveedor": df_ubiproveedor,
    "F_Procesos": df_datos_completos
}

for nombre, df in dataframes.items():
    print(f"\n📄 {nombre} ({df.shape[0]} filas, {df.shape[1]} columnas)")
    display(df.head(3))


In [ ]:
# ============================================
# 5️⃣ RESUMEN BÁSICO DE TODAS LAS TABLAS
# ============================================

def resumen_basico(df):
    return {
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Duplicados": df.duplicated().sum(),
        "Nulos Totales": df.isnull().sum().sum(),
        "Columnas con Nulos": df.isnull().sum()[df.isnull().sum() > 0].to_dict(),
        "Tipos": df.dtypes.astype(str).to_dict()
    }

global_summary = []

for nombre, df in dataframes.items():
    print("\n" + "="*60)
    print(f"TABLA: {nombre}")
    print("="*60)

    resumen = resumen_basico(df)
    global_summary.append({"Tabla": nombre, **resumen})

    print(resumen)
    print("\nPrimeras filas:")
    display(df.head(3))

# Convertir a DataFrame para mejor visualización
global_summary = pd.DataFrame(global_summary)
display(global_summary)


In [ ]:
# ============================================
# 🔍 BLOQUE 1 — ANÁLISIS DE DATOS FALTANTES
# ============================================

!pip install missingno > /dev/null

import missingno as msno

print("============================================")
print("🔍 ANÁLISIS DE DATOS FALTANTES")
print("============================================")

df = df_datos_completos.copy()   # Atajo

# 1) Conteo total de nulos por columna
faltantes = df.isnull().sum().sort_values(ascending=False)
faltantes_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

tabla_faltantes = pd.DataFrame({
    "Nulos": faltantes,
    "% Nulos": faltantes_pct.round(2)
})

print("\n👉 Tabla de columnas con datos faltantes:")
display(tabla_faltantes)

# 2) Cuántas columnas tienen al menos 1 nulo
num_columnas_con_nulos = (df.isnull().sum() > 0).sum()
print(f"\n👉 Columnas con al menos un valor faltante: {num_columnas_con_nulos}")

# 3) Visualización del patrón de nulos
print("\n📊 Mapa de calor de valores faltantes:")
msno.matrix(df)

print("\n📊 Heatmap de correlación de nulos (si aplica):")
msno.heatmap(df)

# 4) Conteo de filas completas vs incompletas
filas_completas = df.dropna().shape[0]
filas_incompletas = df.shape[0] - filas_completas

print(f"\n👉 Filas completas: {filas_completas}")
print(f"👉 Filas con datos faltantes: {filas_incompletas}")


In [ ]:
# ============================================
# 📊 BLOQUE 2 — ANÁLISIS UNIVARIADO
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

print("============================================")
print("📊 ANÁLISIS UNIVARIADO")
print("============================================")

df = df_datos_completos.copy()

# 1) Detectar columnas numéricas y categóricas
numericas = df.select_dtypes(include=['int64', 'float64']).columns
categoricas = df.select_dtypes(include=['object','category']).columns

print("\n👉 Variables numéricas:")
print(list(numericas))

print("\n👉 Variables categóricas:")
print(list(categoricas))

# 2) Resumen estadístico para numéricas
print("\n📈 Resumen estadístico variables numéricas:")
display(df[numericas].describe().T)

# 3) Tablas de frecuencia para categóricas
print("\n📊 Frecuencias de variables categóricas:")
for col in categoricas:
    print(f"\n🔸 Variable: {col}")
    display(df[col].value_counts(dropna=False).to_frame("Frecuencia"))

# 4) Histogramas de todas las variables numéricas
print("\n📈 Histogramas variables numéricas:")
for col in numericas:
    plt.figure(figsize=(7,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Histograma: {col}")
    plt.show()

# 5) Boxplots para detectar outliers
print("\n📦 Boxplots de variables numéricas:")
for col in numericas:
    plt.figure(figsize=(7,3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot: {col}")
    plt.show()


In [ ]:
# ============================================
# 📄 BLOQUE 3 — INFORME DE CALIDAD DE DATOS
# ============================================

df = df_datos_completos.copy()

print("============================================")
print("📄 INFORME DE CALIDAD DE DATOS")
print("============================================")

quality_report = pd.DataFrame({
    "Tipo de Dato": df.dtypes,
    "Valores Únicos": df.nunique(),
    "Nulos (#)": df.isnull().sum(),
    "Nulos (%)": (df.isnull().mean()*100).round(2),
    "Duplicados": df.duplicated().sum()
})

display(quality_report)


In [ ]:
# ============================================
# 🧼 BLOQUE 4 — LIMPIEZA DE VARIABLES CATEGÓRICAS
# ============================================

import numpy as np
import unicodedata

df = df_datos_completos.copy()

def clean_text(x):
    if pd.isnull(x): return x
    x = str(x).strip().upper()
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode()
    if x in ["", "NA", "N/A", "-", "--", "SIN DATOS", "NULL"]:
        return np.nan
    return x

categoricas = df.select_dtypes(include=["object", "category"]).columns

for col in categoricas:
    df[col] = df[col].apply(clean_text)

print("👉 Limpieza categórica completada.")


In [ ]:
# ============================================
# ⚠️ BLOQUE 5 — DETECCIÓN DE OUTLIERS
# ============================================

df = df_datos_completos.copy()

numericas = df.select_dtypes(include=['int64', 'float64']).columns
outliers_resumen = {}

for col in numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)][col].count()

    outliers_resumen[col] = {
        "Outliers IQR (#)": outliers,
        "Límite Inferior": lower,
        "Límite Superior": upper
    }

df_outliers = pd.DataFrame(outliers_resumen).T
print("⚠️ Resumen de outliers detectados:")
display(df_outliers)


In [ ]:
# ============================================
# 🔧 BLOQUE 6 — IMPUTACIÓN DE DATOS FALTANTES
# ============================================

from sklearn.impute import SimpleImputer

df = df_datos_completos.copy()

num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object','category']).columns

# Imputación numérica (mediana)
imputer_num = SimpleImputer(strategy="median")
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# Imputación categórica (moda)
imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

print("✅ Imputación realizada correctamente.")


In [ ]:
# ============================================
# 🎨 BLOQUE 7 — VISUALIZACIONES AVANZADAS
# ============================================

df = df_datos_completos.copy()
numericas = df.select_dtypes(include=['int64','float64']).columns
categoricas = df.select_dtypes(include=['object','category']).columns

import matplotlib.pyplot as plt
import seaborn as sns

# Distribuciones log-transformadas
print("\n📈 Histogramas log-transformados (solo variables > 0):")
for col in numericas:
    if (df[col] > 0).all():
        plt.figure(figsize=(7,4))
        sns.histplot(np.log1p(df[col]), kde=True)
        plt.title(f"Log Transformado: {col}")
        plt.show()

# Pareto categórico (top 10)
print("\n📊 Diagramas de Pareto:")
for col in categoricas:
    plt.figure(figsize=(8,4))
    df[col].value_counts().head(10).plot(kind="bar")
    plt.title(f"Pareto top 10: {col}")
    plt.show()


In [ ]:
# ============================================
# 💾 BLOQUE 8 — EXPORTAR RESULTADOS DEL EDA
# ============================================

df_faltantes = df_datos_completos.isnull().sum().to_frame("Nulos")
df_faltantes["%"] = df_datos_completos.isnull().mean()*100

df_outliers.to_excel("resumen_outliers.xlsx")
df_faltantes.to_excel("reporte_faltantes.xlsx")
quality_report.to_excel("reporte_calidad_datos.xlsx")

print("📁 Archivos exportados correctamente:")
print("- resumen_outliers.xlsx")
print("- reporte_faltantes.xlsx")
print("- reporte_calidad_datos.xlsx")


In [ ]:
# ============================================
# 📝 BLOQUE 9 — REPORTE FINAL CONSOLIDADO
# ============================================

print("============================================")
print("📝 REPORTE FINAL DEL EDA")
print("============================================")

print("\n✔️ Total de filas:", df_datos_completos.shape[0])
print("✔️ Total de columnas:", df_datos_completos.shape[1])

print("\n📌 Variables numéricas:", len(numericas))
print("📌 Variables categóricas:", len(categoricas))

print("\n⚠️ Columnas con nulos antes de imputación:")
display(df_faltantes.sort_values("%", ascending=False))

print("\n📦 Resumen de outliers:")
display(df_outliers)

print("\n📄 Informe de calidad de datos:")
display(quality_report.head())
